**Repaired copy.** The upstream file `LLM TREC.ipynb` (commit 657355688d4b) is cut inside an image output and is not valid JSON.
This file keeps the 7 complete cells (loader, prompts, 1-shot examples, grid-search call). The cells after the cut held only plots and are lost.
Cell outputs are the authors' outputs. They are not results from our runs.

In [1]:
import os
import ray
import random
import datetime

import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as plt

from util import *
from estimator import *
from evaluator import run_grid_search

os.environ.pop('http_proxy', None)

'http://192.168.0.1:8123'

### Parse TREC data

In [2]:
train_data = Path('./data/TREC/train.txt')
test_data = Path('./data/TREC/test.txt')

def parse_TREC_file(file_path):
    class_map = {
        "NUM": "Number",
        "DESC": "Description",
        "ENTY": "Entity",
        "ABBR": "Abbreviation",
        "LOC": "Location",
        "HUM": "Person",
    }
    
    data_list = []
    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()
            if line:  # Check if the line is not empty
                parts = line.split(' ', 1)  # Split the line into two parts based on the first space
                if len(parts) == 2:
                    class_subclass, text = parts
                    class_parts = class_subclass.split(':')
                    if len(class_parts) == 2:
                        class_name, subclass_name = class_parts
                        # Generate a dictionary for the current sample
                        sample_dict = {
                            'id': len(data_list) + 1,  # Using 1-based index for id
                            'input': text,
                            'class': class_map[class_name],
                            'subclass': subclass_name
                        }
                        data_list.append(sample_dict)
                    else:
                        print(f"Ignoring line due to incorrect format: {line}")
                else:
                    print(f"Ignoring line due to incorrect format: {line}")
    
    return data_list


train_list = parse_TREC_file(train_data)
test_list = parse_TREC_file(test_data)

class_labels = ["Number", "Location", "Person", "Description", "Entity", "Abbreviation", "N/A"] # { s['class'] for s in test_list } 
# subclass_labels = { s['subclass'] for s in test_list }


class_labels_to_id = {c_name: i for i, c_name in enumerate(class_labels)}

def class_extractor_fun(text):
    if "number" in text.lower(): 
        return "Number"
    elif "location" in text.lower(): 
        return "Location"
    elif "person" in text.lower(): 
        return "Person"
    elif "description" in text.lower(): 
        return "Description"
    elif "entity" in text.lower(): 
        return "Entity"
    elif "abbreviation" in text.lower(): 
        return "Abbreviation"
    else:
        return "N/A"

## Run a grid search over parameters

In [3]:
results_folder = './data/TREC_RESULTS'

question_to_rewrite = "Classify the questions based on whether their answer type is a Number, Location, Person, Description, Entity, or Abbreviation."

prompt_simple = [
    ["system", "You are a text classifier."],
    ["user", "Classify the questions based on whether their answer type is a Number, Location, Person, Description, Entity, or Abbreviation.\nUse only one word that describes the class in your response.\nThe possible classes are: Number, Location, Person, Description, Entity, and Abbreviation.\nHere is the text: {summary}"]    
]

prompt_instruct = [
    ["system", "You are a text classifier."],
    ["user", "Classify the questions based on whether their answer type is a Number, Location, Person, Description, Entity, or Abbreviation.\nUse only one word that describes the class in your response.\nThe possible classes are: \n- Number: A number is a mathematical concept used to count, measure, or label. It can represent a quantity or position in a sequence. Numbers can be whole (like 5), fractional (like 3.14), negative or complex.\n- Location: A location refers to a specific place or position in space, often described by coordinates or a physical address. It can be a point on a map, a geographical area, or a designated site such as a city, building, or landmark.\n- Person: A person is an individual human being with a distinct identity, personality, and consciousness. It typically refers to a living human, but can also be used to describe fictional characters or personas.\n- Description: A description is a detailed portrayal or account of the characteristics, features, or qualities of something or someone. It aims to provide information that helps identify or understand the subject being described.\n- Entity: An entity is a thing or object that exists as a distinct and discernible unit. It can be concrete (like a physical object) or abstract (like a concept or idea). In various contexts, an entity can refer to a person, organization, concept, or element in a system.\n- Abbreviation: An abbreviation is a shortened form of a word or phrase. It is typically formed by taking the initial letters (or parts) of a longer term to create a more concise representation. Abbreviations are often used to save space, time, or effort in writing or speech.\nHere is the text: {summary}"]    
]

prompt_fewshot = [
    ["system", "You are a text classifier."],
    ["user", "Classify the questions based on whether their answer type is a Number, Location, Person, Description, Entity, or Abbreviation.\nUse only one word that describes the class in your response.\nThe possible classes are: Number, Location, Person, Description, Entity, and Abbreviation. Here are some examples to give you a reference:\n- Description: What 's the Olympic motto ?\n- Location: What country 's capital is Tirana ?\n- Number: How many people in the world speak French ?\n- Person: Who is Snoopy 's arch-enemy ?\n- Abbreviation: What does the abbreviation AIDS stand for ?\n- Entity: What 's the only color Johnny Cash wears on stage ?\nHere is the text to classify: {summary}"]    
]


prompt_types = {"simple": prompt_simple,
               "instruct": prompt_instruct,
               "fewshot": prompt_fewshot}

llms = ['llama3', 'mixtral']#, 'gpt-3.5', 'gpt-4o']
Qs = [30] # number of equivalent but alternative questions compared to original question
temp_questions = [0.] # list(np.arange(0., 1.25, 0.25))  # temperature of the LLM used to produce questions
As = [1] # number of alternative answers for each questions
temp_answers =  [0.] # list(np.arange(0., 1.5, 0.25))  # temperature of the LLM used to produce answers

samples = test_list
sample_ids = [s["id"] for s in samples]

print(f'Num test examples is {len(samples)}')

Num test examples is 500


In [4]:
run_grid_search(samples,
                llms,
                prompt_types,
                Qs,
                temp_questions,
                question_to_rewrite,
                As,
                temp_answers,
                class_labels,
                class_extractor_fun,
                results_folder)

KeyboardInterrupt: 

## Plot qualitative results